# Script to transform ICES in situ data into .parquet tabular files  
- Input file format: ICESinsitu_19XX_19XX.csv
- Output file format: {var}_{year}.parquet
- ICES data have been downloaded via: https://data.ices.dk/view-map. Dataset: Ocean hydrochemistry/Bottle and Low Resolution CTD Data.  
- This script have already been processed for the period 1960-2025 and variables: Oxy, NOx, NH4, PO4, SiO, Chl, Temp, Sal, SPM, pH, TA. 
- Output .parquet files are available in **GoogleDrive/2025_CodeBlue_project/WP4/ICESData_for_validation**

_Authors: Capet A. & Denis P._

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import pyarrow as pa
import pyarrow.parquet as pq
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np

In [95]:
filename = "/hpcperm/bel9956/codeblue/ICESData_1960_2025/ICESinsitu_2020_2025.csv" #input directory with data
outdir = "/hpcperm/bel9956/codeblue/ICESData_for_validation/" #output directory where output parquet files are written

In [87]:
with open(filename) as f:
    for line in f:
        if line.startswith("Cruise,"):
            header = line
            break

    df = pd.read_csv(f, names=header.strip().split(","))  # continue reading

/etc/ecmwf/ssd/ssd1/jupyterhub/bel9956-jupyterhub/tmpdirs/bel9956.14031151/ipykernel_3949397/2379488921.py:7: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f, names=header.strip().split(","))  # continue reading


In [88]:
ICESvardic={
    'yyyy-mm-ddThh:mm:ss.sss':'datetime',
    'Longitude [degrees_east]':'lon',
    'Latitude [degrees_north]':'lat',
    'Secchi Depth [m]':'sec',
    'Depth (ADEPZZ01_ULAA) [m]':'depth',
    'QV:ODV:Depth (ADEPZZ01_ULAA) [m]':'depth_qv',
    'Temperature (TEMPPR01_UPAA) [degC]':'temp',
    'QV:ODV:Temperature (TEMPPR01_UPAA) [degC]':'temp_qv',
    'Salinity (PSALPR01_UUUU) [dmnless]':'sal',
    'QV:ODV:Salinity (PSALPR01_UUUU) [dmnless]':'sal_qv',
    'Oxygen (DOXYZZXX_UMLL) [ml/l]':'oxy',
    'QV:ODV:Oxygen (DOXYZZXX_UMLL) [ml/l]':'oxy_qv',
    'Phosphate (PHOSZZXX_UPOX) [umol/l]':'po4',
    'QV:ODV:Phosphate (PHOSZZXX_UPOX) [umol/l]':'po4_qv',
    'Silicate (SLCAZZXX_UPOX) [umol/l]':'sio',
    'QV:ODV:Silicate (SLCAZZXX_UPOX) [umol/l]':'sio_qv',
    'Nitrate + Nitrite (NTRZZZXX_UPOX) [umol/l]':'nox',
    'QV:ODV:Nitrate + Nitrite (NTRZZZXX_UPOX) [umol/l]':'nox_qv',
    'Ammonium (AMONZZXX_UPOX) [umol/l]':'nh4',
    'QV:ODV:Ammonium (AMONZZXX_UPOX) [umol/l]':'nh4_qv',
    'pH (PHXXZZXX_UUPH) [pH units]':'ph',
    'QV:ODV:pH (PHXXZZXX_UUPH) [pH units]':'ph_qv',
    'Total Alkalinity (ALKYZZXX_MEQL) [mEq/l]':'talk',
    'QV:ODV:Total Alkalinity (ALKYZZXX_MEQL) [mEq/l]':'talk_qv',
    'Chlorophyll a (CPHLZZXX_UGPL) [ug/l]':'chl',
    'QV:ODV:Chlorophyll a (CPHLZZXX_UGPL) [ug/l]':'chl_qv',
    'Suspended particulate material (TSEDZZZZ_UMGL) [mg/l]':'spm',
    'QV:ODV:Suspended particulate material (TSEDZZZZ_UMGL) [mg/l]':'spm_qv',
}

In [89]:
for c in df.columns:
    if c in ICESvardic:
        if ICESvardic[c] is not None:
            df.rename(columns={c:ICESvardic[c]}, inplace=True)
    else:
        df.drop(columns=[c], inplace=True)

df = df.dropna(subset=['datetime', 'lon', 'lat'])
df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce')
df

,datetime,lon,lat,sec,depth,depth_qv,temp,temp_qv,sal,sal_qv,...,nh4,nh4_qv,ph,ph_qv,talk,talk_qv,chl,chl_qv,spm,spm_qv
0,2020-01-21 14:35:00+00:00,3.120000,51.361700,NaN,3.0,0,NaN,1,31.7239,0,...,4.400,0,NaN,1,NaN,1,5.60,0,NaN,1
1,2020-01-22 14:06:00+00:00,2.808300,51.416700,NaN,3.0,0,NaN,1,33.7285,0,...,NaN,1,8.18,0,NaN,1,2.90,0,NaN,1
2,2020-01-23 10:09:00+00:00,2.350000,51.458300,NaN,3.0,0,NaN,1,34.8874,0,...,0.300,0,8.21,0,NaN,1,2.00,0,NaN,1
3,2020-01-24 05:40:00+00:00,2.666700,51.168300,NaN,3.0,0,NaN,1,33.3261,0,...,4.300,0,8.46,0,NaN,1,0.81,0,NaN,1
4,2020-02-19 12:00:00+00:00,3.120000,51.361700,NaN,3.0,0,NaN,1,33.3446,0,...,1.300,0,7.98,0,NaN,1,4.50,0,NaN,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
280921,2025-07-02 14:30:00+00:00,20.706673,55.404999,NaN,5.0,1,14.56,1,7.4800,1,...,NaN,1,8.24,1,NaN,1,NaN,1,NaN,1
280922,2025-07-02 14:30:00+00:00,20.706673,55.404999,NaN,10.0,1,13.88,1,7.4800,1,...,NaN,1,8.18,1,NaN,1,NaN,1,NaN,1
280923,2025-07-02 14:30:00+00:00,20.706673,55.404999,NaN,20.0,1,13.79,1,7.4800,1,...,NaN,1,8.13,1,NaN,1,NaN,1,NaN,1
280924,2025-07-02 14:30:00+00:00,20.706673,55.404999,NaN,30.0,1,13.65,1,7.4900,1,...,NaN,1,8.09,1,NaN,1,NaN,1,NaN,1


In [90]:
### Check if variable exists in the Dataframe
def onevar(df, var):

    if var not in df.columns:
        print(f"Variable '{var}' not found in DataFrame.")
        return None
    df = df[['datetime','lon','lat','depth', var, var+'_qv']].dropna()
    df = df[df[var+'_qv'] <= 1]

    return df

In [91]:
### Visualisation plots of the available data
def mapvar(df, var, figname=None):
    dfv = onevar(df, var)

    # Map of stations 2000-2005
    plt.figure(figsize=(12, 6))

    ax= plt.subplot(2,3,1,projection=ccrs.PlateCarree())
    ax.set_extent([-10, 15, 48, 60], crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND)
    ax.add_feature(cfeature.COASTLINE)  
    ax.scatter(dfv['lon'], dfv['lat'], alpha=0.5, s=10, c='blue')
    ax.set_title('%s Locations (%s-%s)'%(var, dfv['datetime'].dt.year.min(),dfv['datetime'].dt.year.max()))

    ax = plt.subplot(2,3,4)
    ax.hist(dfv['datetime'].dt.month, bins=range(1, 14), edgecolor='black')
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
    ax.set_title('Number of datas per Month')
    ax.set_xlabel('Month')


    # monthly box plot with number of data per months
    ax = plt.subplot(1,3,2)
    ax.boxplot([dfv[dfv['datetime'].dt.month == m][var] for m in range(1, 13)], labels=['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
    ax.set_title('Monthly Box Plot of %s'%var)
    ax.set_xlabel('Month')
    ax.set_ylabel(var)  
    ax.set_ylim(0, dfv[var].quantile(0.95))  # Limit y-axis to 95th percentile to avoid outliers dominating the plot


    # monthly box plot with number of data per months
    ax = plt.subplot(1,3,3)
    ax.hist(dfv[var], bins=np.linspace(0, dfv[var].quantile(0.95), 30), edgecolor='black')
    
    ax.set_title('Total number of data: %d'%len(dfv))

    ax.annotate('Mean: %.2f'%dfv[var].mean(), xy=(0.7, 0.95), xycoords='axes fraction')
    ax.annotate('Std: %.2f'%dfv[var].std(), xy=(0.7, 0.85), xycoords='axes fraction')        
    ax.annotate('p05: %.2f'%dfv[var].quantile(0.05), xy=(0.7, 0.80), xycoords='axes fraction ')
    ax.annotate('p25: %.2f'%dfv[var].quantile(0.25), xy=(0.7, 0.75), xycoords='axes fraction ')
    ax.annotate('Median: %.2f'%dfv[var].median(), xy=(0.7, 0.70), xycoords='axes fraction ')
    ax.annotate('p75: %.2f'%dfv[var].quantile(0.75), xy=(0.7, 0.65), xycoords='axes fraction ')
    ax.annotate('p95: %.2f'%dfv[var].quantile(0.95), xy=(0.7, 0.60), xycoords='axes fraction ')

    plt.tight_layout()
    # if figname is not None:
        # plt.savefig(figname, dpi=300)
    plt.show()

In [93]:
for v in ['oxy', 'nox', 'nh4', 'po4', 'sio', 'chl', 'temp', 'sal', 'spm', 'ph', 'talk']:
    # if True:
        # mapvar(df, v, figname='./%s_map.png'%v)
    if True:
        dfv = onevar(df, v)
        if dfv is None:
            continue  # skip missing variables
        
        # 1 Parquet file per year
        for year, df_year in dfv.groupby(dfv['datetime'].dt.year): 
            table = pa.Table.from_pandas(df_year, preserve_index=False)
            pq.write_table(table, outdir +f'{v}_{year}.parquet')